# How to Fine-Tune LLMs with LoRA Adapters using Hugging Face TRL

This notebook demonstrates how to efficiently fine-tune large language models using LoRA (Low-Rank Adaptation) adapters. LoRA is a parameter-efficient fine-tuning technique that:
- Freezes the pre-trained model weights
- Adds small trainable rank decomposition matrices to attention layers
- Typically reduces trainable parameters by ~90%
- Maintains model performance while being memory efficient

We'll cover:
1. Setup development environment and LoRA configuration
2. Create and prepare the dataset for adapter training
3. Fine-tune using `trl` and `SFTTrainer` with LoRA adapters
4. Test the model and merge adapters (optional)


## 1. Setup development environment

Our first step is to install Hugging Face Libraries and Pyroch, including trl, transformers and datasets. If you haven't heard of trl yet, don't worry. It is a new library on top of transformers and datasets, which makes it easier to fine-tune, rlhf, align open LLMs.


In [1]:
# Install the requirements in Google Colab
# !pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face

from huggingface_hub import login
import yaml

# Load the config file
with open("../../config.yaml", "r") as file:
    config = yaml.safe_load(file)

# Extract the token
hf_token = config.get("huggingface_hub", {}).get("token")

if hf_token:
    # Log in to Hugging Face Hub
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Access token not found in config.yaml!")
#login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

Successfully logged in to Hugging Face!


## 2. Load the dataset

In [2]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})

In [3]:
# Inspect the first training example
print("First Example in Training Split:")
print(dataset["train"][0])  # Adjust 'train' if your dataset split is different

First Example in Training Split:
{'full_topic': 'Travel/Vacation destinations/Beach resorts', 'messages': [{'content': 'Hi there', 'role': 'user'}, {'content': 'Hello! How can I help you today?', 'role': 'assistant'}, {'content': "I'm looking for a beach resort for my next vacation. Can you recommend some popular ones?", 'role': 'user'}, {'content': "Some popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.", 'role': 'assistant'}, {'content': 'That sounds great. Are there any resorts in the Caribbean that are good for families?', 'role': 'user'}, {'content': 'Yes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer a range of activities and amenities suitable for all ages.', 'role': 'assistant'}, {'content': "Okay, I'll look into those. Thanks for the recommendations!", 'role': 'user'}, {'content': "You're welcome. I hope you f

In [4]:
# Import necessary libraries
from datasets import load_dataset, Dataset, load_from_disk
from datasets import DatasetDict
# Function to process dataset into TRL chat template format
def process_dataset_to_chat_template(dataset):
    """
    Processes the dataset to convert it into the TRL chat template format.

    Args:
        dataset: The dataset to process.
    Returns:
        A Dataset in the required chat template format.
    """
    chat_data = []

    # Iterate through each example in the dataset
    for example in dataset:
        # Access the 'messages' field
        messages = example.get("messages", [])

        # Initialize an empty chat template
        chat_template = ""

        # Iterate through messages and build the conversation template
        for msg in messages:
            role = msg.get("role", "")
            content = msg.get("content", "")
            
            if role and content:
                if role == "user":
                    chat_template += f"<|im_start|>user\n{content}<|im_end|>"
                elif role == "assistant":
                    chat_template += f"<|im_start|>assistant\n{content}<|im_end|>"

        # Append the chat template if not empty
        if chat_template:
            chat_data.append({"text": chat_template})

    if not chat_data:
        raise ValueError("No valid examples found in the dataset! Check the dataset structure and keys.")

    # Convert to Hugging Face Dataset
    return Dataset.from_list(chat_data)

# Process the dataset
if "train" in dataset:
    processed_train = process_dataset_to_chat_template(dataset["train"])
else:
    raise ValueError("Dataset does not have a 'train' split! Check the dataset structure.")

# Wrap the processed dataset into a DatasetDict
processed_dataset = DatasetDict({"train": processed_train})

if "test" in dataset:
    processed_test = process_dataset_to_chat_template(dataset["test"])
else:
    raise ValueError("Dataset does not have a 'test' split! Check the dataset structure.")

# Wrap the processed dataset into a DatasetDict
processed_dataset = DatasetDict({"train": processed_train,
                                "test": processed_test})

# Inspect the processed dataset
print("Sample Processed Dataset Entry:")
if len(processed_dataset["train"]) > 0:
    print(processed_dataset["train"][0])
else:
    print("Processed dataset is empty!")

# Save the processed dataset
processed_dataset.save_to_disk("processed_chat_template_dataset")

Sample Processed Dataset Entry:
{'text': "<|im_start|>user\nHi there<|im_end|><|im_start|>assistant\nHello! How can I help you today?<|im_end|><|im_start|>user\nI'm looking for a beach resort for my next vacation. Can you recommend some popular ones?<|im_end|><|im_start|>assistant\nSome popular beach resorts include Maui in Hawaii, the Maldives, and the Bahamas. They're known for their beautiful beaches and crystal-clear waters.<|im_end|><|im_start|>user\nThat sounds great. Are there any resorts in the Caribbean that are good for families?<|im_end|><|im_start|>assistant\nYes, the Turks and Caicos Islands and Barbados are excellent choices for family-friendly resorts in the Caribbean. They offer a range of activities and amenities suitable for all ages.<|im_end|><|im_start|>user\nOkay, I'll look into those. Thanks for the recommendations!<|im_end|><|im_start|>assistant\nYou're welcome. I hope you find the perfect resort for your vacation.<|im_end|>"}


Saving the dataset (0/1 shards):   0%|          | 0/2260 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/119 [00:00<?, ? examples/s]

In [5]:
# Load the processed dataset
dataset_path = "processed_chat_template_dataset"
processed_dataset = load_from_disk(dataset_path)
# Verify dataset splits
train_dataset = processed_dataset["train"]
eval_dataset = processed_dataset["test"]  # Ensure the dataset has a 'test' split

## 3. Fine-tune LLM using `trl` and the `SFTTrainer` with LoRA

The [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) from `trl` provides integration with LoRA adapters through the [PEFT](https://huggingface.co/docs/peft/en/index) library. Key advantages of this setup include:

1. **Memory Efficiency**: 
   - Only adapter parameters are stored in GPU memory
   - Base model weights remain frozen and can be loaded in lower precision
   - Enables fine-tuning of large models on consumer GPUs

2. **Training Features**:
   - Native PEFT/LoRA integration with minimal setup
   - Support for QLoRA (Quantized LoRA) for even better memory efficiency

3. **Adapter Management**:
   - Adapter weight saving during checkpoints
   - Features to merge adapters back into base model

We'll use LoRA in our example, which combines LoRA with 4-bit quantization to further reduce memory usage without sacrificing performance. The setup requires just a few configuration steps:
1. Define the LoRA configuration (rank, alpha, dropout)
2. Create the SFTTrainer with PEFT config
3. Train and save the adapter weights


In [6]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-smoltalk3e"
finetune_tags = ["smol-course", "module_1"]

The `SFTTrainer`  supports a native integration with `peft`, which makes it super easy to efficiently tune LLMs using, e.g. LoRA. We only need to create our `LoraConfig` and provide it to the trainer.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Define LoRA parameters for finetuning</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p> 
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the general parameters for an abitrary finetune</p>
    <p>🐕 Adjust the parameters and review in weights & biases.</p>
    <p>🦁 Adjust the parameters and show change in inference results.</p>
</div>

In [7]:
from peft import LoraConfig

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 6
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

Before we can start our training we need to define the hyperparameters (`TrainingArguments`) we want to use.

In [8]:
# Training configuration
# Hyperparameters based on QLoRA paper recommendations
args = SFTConfig(
    # Output settings
    output_dir=finetune_name,  # Directory to save model checkpoints
    # Training duration
    num_train_epochs=3,  # Number of training epochs
    # Batch size settings
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=2,  # Accumulate gradients for larger effective batch
    # Memory optimization
    gradient_checkpointing=True,  # Trade compute for memory savings
    # Optimizer settings
    optim="adamw_torch_fused",  # Use fused AdamW for efficiency
    learning_rate=2e-4,  # Learning rate (QLoRA paper)
    max_grad_norm=0.3,  # Gradient clipping threshold
    # Learning rate schedule
    warmup_ratio=0.03,  # Portion of steps for warmup
    lr_scheduler_type="constant",  # Keep learning rate constant after warmup
    # Logging and saving
    logging_steps=10,  # Log metrics every N steps
    save_strategy="epoch",  # Save checkpoint every epoch
    # Precision settings
    bf16=True,  # Use bfloat16 precision
    # Integration settings
    push_to_hub=False,  # Don't push to HuggingFace Hub
    report_to=None,  # Disable external logging
)

We now have every building block we need to create our `SFTTrainer` to start then training our model.

In [9]:
max_seq_length = 1512  # max sequence length for model and packing of the dataset

# Create SFTTrainer with LoRA configuration
from trl import SFTTrainer

# Tokenize and preprocess train and eval datasets
train_dataset = train_dataset.map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        add_special_tokens=False,  # Special tokens handled manually
    ),
    batched=True,
)

eval_dataset = eval_dataset.map(
    lambda x: tokenizer(
        x["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        add_special_tokens=False,  # Special tokens handled manually
    ),
    batched=True,
)

# Initialize SFTTrainer without dataset_kwargs
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    peft_config=peft_config,  # LoRA configuration
    eval_dataset=eval_dataset,
)



Map:   0%|          | 0/119 [00:00<?, ? examples/s]

Start training our model by calling the `train()` method on our `Trainer` instance. This will start the training loop and train our model for 3 epochs. Since we are using a PEFT method, we will only save the adapted model weights and not the full model.

In [10]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,4.929000
20,3.967400
30,3.431300
40,3.092500
50,2.798400
60,2.521700
70,2.259500
80,2.052800
90,1.917100
100,1.829200


The training with Flash Attention for 3 epochs with a dataset of 15k samples took 4:14:36 on a `g5.2xlarge`. The instance costs `1.21$/h` which brings us to a total cost of only ~`5.3$`.



### Merge LoRA Adapter into the Original Model

When using LoRA, we only train adapter weights while keeping the base model frozen. During training, we save only these lightweight adapter weights (~2-10MB) rather than a full model copy. However, for deployment, you might want to merge the adapters back into the base model for:

1. **Simplified Deployment**: Single model file instead of base model + adapters
2. **Inference Speed**: No adapter computation overhead
3. **Framework Compatibility**: Better compatibility with serving frameworks


In [11]:
from peft import AutoPeftModelForCausalLM


# Load PEFT model on CPU
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# Merge LoRA and base model and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained(
    args.output_dir, safe_serialization=True, max_shard_size="2GB"
)

## 3. Test Model and run Inference

After the training is done we want to test our model. We will load different samples from the original dataset and evaluate the model on those samples, using a simple loop and accuracy as our metric.



<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Load LoRA Adapter</h2>
    <p>Use what you learnt from the ecample note book to load your trained LoRA adapter for inference.</p> 
</div>

In [12]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

In [13]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

# Define the prompts
prompts = [
    "What is the capital of Germany? Explain why that's the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]

# Load Model with PEFT adapter
tokenizer = AutoTokenizer.from_pretrained(finetune_name)
model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name, device_map="auto", torch_dtype=torch.float16
)
pipe = pipeline(
    "text-generation", model=merged_model, tokenizer=tokenizer, device=device
)

Lets test some prompt samples and see how the model performs.

In [14]:
def test_inference(prompt, chat_template="{% for message in messages %}{{ message.role }}: {{ message.content }}\n{% endfor %}"):
    # Apply chat template
    formatted_prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        chat_template=chat_template,  # Provide the chat template
        tokenize=False,
        add_generation_prompt=True,
    )
    # Generate output with adjusted max length
    outputs = pipe(
        formatted_prompt,
        max_new_tokens=100,  # Maximum tokens to generate
        pad_token_id=pipe.tokenizer.pad_token_id,  # Handle padding if necessary
    )
    return outputs[0]["generated_text"][len(formatted_prompt) :].strip()


for prompt in prompts:
    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

    prompt:
What is the capital of Germany? Explain why that's the case and if it was different in the past?
    response:
Alice: Germany has a capital city, Berlin, which is located in the southern part of the country. It's a bustling city with many businesses and people.

Bob: That makes sense. What's the main language spoken in Germany?

Alice: German is the main language in Germany, but other languages like French, Italian, and English are also spoken.

Bob: I see. So, Germany is a multilingual country.

Alice: Yes, that's right
--------------------------------------------------
    prompt:
Write a Python function to calculate the factorial of a number.
    response:
## 2. Write a Python function to calculate the factorial of a number.

The factorial of a number is the product of all the numbers from 1 to that number. For example, if the number is 3, the factorial is 3 * 2 * 1 = 6.

The factorial of a number can be calculated using the following formula:

`n! = n * (n-1)!`

For exa